# 📊 Dashboard Interactivo de EDA + CRISP-DM
## Análisis de Rotación de Personal — IBM Human Resources Dataset

Este dashboard presenta un análisis exploratorio completo del dataset de **Recursos Humanos de IBM**,
siguiendo la metodología **CRISP-DM** y técnicas de **Análisis Exploratorio de Datos (EDA)**.

### Contexto del proyecto
IBM busca comprender los factores que influyen en la **rotación voluntaria de empleados** (attrition),
con el objetivo de diseñar estrategias de retención de talento más efectivas.

### Dataset
- **Fuente:** IBM HR Analytics Employee Attrition & Performance
- **Filas:** 1,470 registros de empleados
- **Columnas:** 36 variables (demográficas, laborales, de satisfacción y compensación)
- **Variable objetivo:** `Rotacion` (Yes / No)

---
> **Analista:** Área de People Analytics — IBM Colombia  
> **Metodología:** CRISP-DM  
> **Herramientas:** Python · Pandas · Plotly · ipywidgets


# 1. Instalación e importación de librerías

En esta sección instalamos e importamos las librerías necesarias.

## Explicación

- `pandas`: permite trabajar con tablas de datos.
- `numpy`: permite realizar cálculos numéricos.
- `plotly.express`: permite crear gráficos interactivos.
- `ipywidgets`: permite crear botones, listas desplegables y controles interactivos.
- `google.colab.files`: permite subir archivos desde el computador.
- `IPython.display`: permite mostrar HTML, Markdown y widgets dentro del notebook.


In [6]:
# INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# IBM HR Dashboard — Recursos Humanos
!pip -q install plotly ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output
from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ Librerías importadas correctamente.")
print("📊 Dataset: IBM HR Analytics — Rotación de Personal")


✅ Librerías importadas correctamente.
📊 Dataset: IBM HR Analytics — Rotación de Personal


# 2. Carga del Dataset IBM HR

El dataset contiene **1,470 registros** de empleados de IBM con **36 variables** que incluyen:

| Categoría | Variables destacadas |
|-----------|----------------------|
| **Demográficas** | Edad, Género, Estado Civil, Campo de Educación |
| **Laborales** | Departamento, Rol Laboral, Nivel Laboral, Años en la empresa |
| **Compensación** | Ingreso Mensual, Tarifa Diaria, Tarifa por Hora, Aumento % Salarial |
| **Satisfacción** | Satisfacción Laboral, Satisfacción Ambiental, Satisfacción Relaciones |
| **Variable objetivo** | **Rotacion** (Yes = rotó / No = se quedó) |

> ⚠️ El dataset **no tiene valores nulos ni duplicados**. Es un dataset limpio y listo para análisis.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd

# Carga del Dataset

df = pd.read_csv('/content/drive/MyDrive/Bootcamp_ADA_702/Proyecto_final/Dataset/data_set_IBM/recursos_humanos_ibm.csv', sep=',', index_col=0)
df

print(f'Dataset cargado correctamente.')
print(f'Filas    : {df.shape[0]:,}')
print(f'Columnas : {df.shape[1]}')


Dataset cargado correctamente.
Filas    : 1,470
Columnas : 35


# 3. Limpieza inicial automática

Antes de construir el dashboard, hacemos una limpieza básica de nombres de columnas.

## ¿Por qué es importante?

Muchos datasets tienen columnas con espacios, mayúsculas, guiones o caracteres incómodos.

Por ejemplo:

- `Customer Name` se convierte en `customer_name`
- `Order-Date` se convierte en `order_date`
- `Total Sales` se convierte en `total_sales`

Esto facilita escribir código y evita errores.


In [9]:
# LIMPIEZA INICIAL DE NOMBRES DE COLUMNAS

df_original = df.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(".", "_", regex=False)
)

print("Columnas después de la limpieza:")
display(pd.DataFrame({"columnas": df.columns}))


Columnas después de la limpieza:


,columnas
0,edad
1,rotacion
2,viajenegocios
3,tarifadiaria
4,departamento
5,distanciacasa
6,educacion
7,campoeducacion
8,cantidadempleados
9,numeroempleado


# 4. Detección automática de tipos de variables

En esta sección detectamos automáticamente:

- Variables numéricas.
- Variables categóricas.
- Posibles variables de fecha.

## Explicación

Esto permite que el dashboard se adapte a casi cualquier dataset.

- Las variables numéricas se usan para histogramas, boxplots, correlaciones y scatter plots.
- Las variables categóricas se usan para gráficos de barras y análisis por grupos.
- Las variables de fecha se usan para análisis temporal.


In [10]:
# DETECCIÓN AUTOMÁTICA DE VARIABLES

# Intentamos convertir columnas tipo texto a fecha cuando sea posible
posibles_fechas = []

for col in df.columns:
    if df[col].dtype == "object":
        try:
            convertida = pd.to_datetime(df[col], errors="coerce")
            porcentaje_fechas_validas = convertida.notna().mean()

            # Si más del 70% de los valores parecen fechas, la convertimos
            if porcentaje_fechas_validas >= 0.70:
                df[col] = convertida
                posibles_fechas.append(col)
        except:
            pass

variables_numericas = df.select_dtypes(include=np.number).columns.tolist()
variables_categoricas = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
variables_fecha = df.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

print("Variables numéricas:")
print(variables_numericas)

print("\nVariables categóricas:")
print(variables_categoricas)

print("\nVariables de fecha detectadas:")
print(variables_fecha)


Variables numéricas:
['edad', 'tarifadiaria', 'distanciacasa', 'educacion', 'cantidadempleados', 'numeroempleado', 'satisfaccionambiente', 'tarifaporhora', 'involucramientolaboral', 'nivellaboral', 'satisfaccionlaboral', 'ingresomensual', 'tarifamensual', 'numempresastrabajadas', 'aumentoporcentualsalario', 'evaluacionrendimiento', 'satisfaccionrelaciones', 'horasestandar', 'nivelopcionesacciones', 'aniostotalestrabajo', 'vecesentrenamientoultimoanio', 'equilibriotrabajovida', 'aniosenempresa', 'aniosenrolactual', 'aniosdesdeultimapromocion', 'anioscongerenteactual']

Variables categóricas:
['rotacion', 'viajenegocios', 'departamento', 'campoeducacion', 'genero', 'rollaboral', 'estadocivil', 'mayorde18', 'horasextra']

Variables de fecha detectadas:
[]


/tmp/ipykernel_1006/226560178.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_1006/226560178.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_1006/226560178.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_1006/226560178.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consi

# 5. Tarjetas de resumen del dataset

Esta sección crea métricas generales, también conocidas como **KPIs**.

## Métricas incluidas

- Número de filas.
- Número de columnas.
- Cantidad de valores nulos.
- Porcentaje total de valores nulos.
- Cantidad de filas duplicadas.
- Número de variables numéricas.
- Número de variables categóricas.
- Número de variables de fecha.

## Explicación

Estas tarjetas ayudan al público a entender rápidamente el tamaño y la calidad general del dataset.


In [11]:
# TARJETAS DE RESUMEN DEL DATASET

def mostrar_tarjetas_resumen(dataframe):
    filas = dataframe.shape[0]
    columnas = dataframe.shape[1]
    total_celdas = dataframe.shape[0] * dataframe.shape[1]
    nulos = dataframe.isnull().sum().sum()
    porcentaje_nulos = (nulos / total_celdas) * 100 if total_celdas > 0 else 0
    duplicados = dataframe.duplicated().sum()

    html = f'''
    <div style="display:grid; grid-template-columns: repeat(4, 1fr); gap:15px; font-family:Arial;">
        <div style="background:#f2f4f8; padding:18px; border-radius:12px; text-align:center;">
            <h3>Filas</h3>
            <h2>{filas:,}</h2>
        </div>
        <div style="background:#f2f4f8; padding:18px; border-radius:12px; text-align:center;">
            <h3>Columnas</h3>
            <h2>{columnas:,}</h2>
        </div>
        <div style="background:#fff3cd; padding:18px; border-radius:12px; text-align:center;">
            <h3>Valores nulos</h3>
            <h2>{nulos:,}</h2>
            <p>{porcentaje_nulos:.2f}% del total</p>
        </div>
        <div style="background:#f8d7da; padding:18px; border-radius:12px; text-align:center;">
            <h3>Duplicados</h3>
            <h2>{duplicados:,}</h2>
        </div>
        <div style="background:#d1ecf1; padding:18px; border-radius:12px; text-align:center;">
            <h3>Numéricas</h3>
            <h2>{len(variables_numericas)}</h2>
        </div>
        <div style="background:#d4edda; padding:18px; border-radius:12px; text-align:center;">
            <h3>Categóricas</h3>
            <h2>{len(variables_categoricas)}</h2>
        </div>
        <div style="background:#e2e3e5; padding:18px; border-radius:12px; text-align:center;">
            <h3>Fechas</h3>
            <h2>{len(variables_fecha)}</h2>
        </div>
        <div style="background:#e8daef; padding:18px; border-radius:12px; text-align:center;">
            <h3>Memoria</h3>
            <h2>{dataframe.memory_usage(deep=True).sum() / 1024**2:.2f} MB</h2>
        </div>
    </div>
    '''

    display(HTML(html))

mostrar_tarjetas_resumen(df)


# 6. Vista interactiva de datos

Esta sección permite explorar una muestra del dataset.

## Explicación del widget

El control `IntSlider` permite seleccionar cuántas filas queremos visualizar.

Esto es útil cuando el dataset es grande y no queremos mostrarlo completo.


In [12]:
# VISTA INTERACTIVA DEL DATASET

slider_filas = widgets.IntSlider(
    value=5,
    min=5,
    max=min(100, len(df)),
    step=5,
    description="Filas:"
)

salida_vista = widgets.Output()

def actualizar_vista(change=None):
    with salida_vista:
        clear_output()
        display(df.head(slider_filas.value))

slider_filas.observe(actualizar_vista, names="value")

display(slider_filas, salida_vista)
actualizar_vista()


IntSlider(value=5, description='Filas:', min=5, step=5)

Output()

# 7. Análisis de calidad de datos

Esta sección analiza valores nulos y duplicados.

## ¿Por qué importa?

La calidad de datos es una parte esencial de CRISP-DM, especialmente en la fase de **Data Preparation**.

Datos faltantes, duplicados o inconsistentes pueden afectar las conclusiones del análisis.


In [13]:

#Tabla de datos nulos
def tabla_valores_nulos(dataframe):
    nulos = dataframe.isnull().sum()
    porcentaje = dataframe.isnull().mean() * 100

    resultado = pd.DataFrame({
        "columna": dataframe.columns,
        "valores_nulos": nulos.values,
        "porcentaje_nulos": porcentaje.values
    })

    resultado = resultado.sort_values("porcentaje_nulos", ascending=False)
    return resultado

tabla_nulos = tabla_valores_nulos(df)
display(tabla_nulos)


,columna,valores_nulos,porcentaje_nulos
0,edad,0,0.00
1,rotacion,0,0.00
2,viajenegocios,0,0.00
3,tarifadiaria,0,0.00
4,departamento,0,0.00
5,distanciacasa,0,0.00
6,educacion,0,0.00
7,campoeducacion,0,0.00
8,cantidadempleados,0,0.00
9,numeroempleado,0,0.00


In [14]:
# GRÁFICO INTERACTIVO DE VALORES NULOS

tabla_nulos_grafico = tabla_nulos[tabla_nulos["valores_nulos"] > 0]

if len(tabla_nulos_grafico) > 0:
    fig = px.bar(
        tabla_nulos_grafico,
        x="porcentaje_nulos",
        y="columna",
        orientation="h",
        title="Porcentaje de valores nulos por columna",
        labels={
            "porcentaje_nulos": "% de valores nulos",
            "columna": "Columna"
        }
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
else:
    print("No se encontraron valores nulos en el dataset.")


No se encontraron valores nulos en el dataset.


# 8. Dashboard interactivo para variables numéricas

Esta sección permite seleccionar una variable numérica y generar automáticamente:

- Histograma.
- Boxplot.
- Estadísticas descriptivas.
- Conteo de valores nulos.

## Explicación

El estudiante puede cambiar la columna desde una lista desplegable y el gráfico se actualiza automáticamente.


In [15]:
# DASHBOARD INTERACTIVO PARA VARIABLES NUMÉRICAS

if len(variables_numericas) > 0:
    selector_numerica = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[0],
        description="Variable:"
    )

    salida_numerica = widgets.Output()

    def dashboard_numerico(change=None):
        with salida_numerica:
            clear_output()

            col = selector_numerica.value

            display(Markdown(f"## Análisis de la variable numérica: `{col}`"))

            resumen = df[col].describe().to_frame(name="valor")
            display(resumen)

            print(f"Valores nulos: {df[col].isnull().sum()}")
            print(f"Valores únicos: {df[col].nunique()}")

            fig_hist = px.histogram(
                df,
                x=col,
                nbins=30,
                title=f"Distribución de {col}",
                marginal="box"
            )
            fig_hist.show()

            fig_box = px.box(
                df,
                y=col,
                title=f"Boxplot de {col}"
            )
            fig_box.show()

    selector_numerica.observe(dashboard_numerico, names="value")

    display(selector_numerica, salida_numerica)
    dashboard_numerico()
else:
    print("No se encontraron variables numéricas en el dataset.")


Dropdown(description='Variable:', options=('edad', 'tarifadiaria', 'distanciacasa', 'educacion', 'cantidadempl…

Output()

# 9. Dashboard interactivo para variables categóricas

Esta sección permite seleccionar una variable categórica y visualizar sus categorías más frecuentes.

## Controles

- Selector de columna categórica.
- Selector del número de categorías a mostrar.

## Explicación

Este análisis es útil para responder preguntas como:

- ¿Cuáles son las categorías más comunes?
- ¿Hay categorías dominantes?
- ¿Existen valores mal escritos o inconsistentes?


In [16]:
# DASHBOARD INTERACTIVO PARA VARIABLES CATEGÓRICAS

if len(variables_categoricas) > 0:
    selector_categorica = widgets.Dropdown(
        options=variables_categoricas,
        value=variables_categoricas[0],
        description="Variable:"
    )

    slider_top = widgets.IntSlider(
        value=10,
        min=3,
        max=30,
        step=1,
        description="Top N:"
    )

    salida_categorica = widgets.Output()

    def dashboard_categorico(change=None):
        with salida_categorica:
            clear_output()

            col = selector_categorica.value
            top_n = slider_top.value

            display(Markdown(f"## Análisis de la variable categórica: `{col}`"))

            conteo = df[col].value_counts(dropna=False).head(top_n).reset_index()
            conteo.columns = [col, "frecuencia"]
            conteo["porcentaje"] = conteo["frecuencia"] / len(df) * 100

            display(conteo)

            fig = px.bar(
                conteo,
                x="frecuencia",
                y=col,
                orientation="h",
                title=f"Top {top_n} categorías de {col}",
                text="porcentaje"
            )
            fig.update_traces(texttemplate="%{text:.1f}%")
            fig.update_layout(yaxis={"categoryorder": "total ascending"})
            fig.show()

    selector_categorica.observe(dashboard_categorico, names="value")
    slider_top.observe(dashboard_categorico, names="value")

    display(widgets.HBox([selector_categorica, slider_top]), salida_categorica)
    dashboard_categorico()
else:
    print("No se encontraron variables categóricas en el dataset.")


Output()

# 10. Análisis interactivo de relaciones entre variables

Esta sección permite analizar relaciones entre variables numéricas usando un gráfico de dispersión.

## Controles

- Variable X.
- Variable Y.
- Variable categórica para colorear los puntos, opcional.

## ¿Para qué sirve?

Este gráfico ayuda a detectar:

- Relaciones positivas.
- Relaciones negativas.
- Grupos o segmentos.
- Valores atípicos.
- Patrones visuales entre variables.


In [17]:
# SCATTER PLOT INTERACTIVO

if len(variables_numericas) >= 2:
    selector_x = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[0],
        description="Eje X:"
    )

    selector_y = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[1],
        description="Eje Y:"
    )

    opciones_color = ["Sin color"] + variables_categoricas
    selector_color = widgets.Dropdown(
        options=opciones_color,
        value="Sin color",
        description="Color:"
    )

    salida_scatter = widgets.Output()

    def dashboard_scatter(change=None):
        with salida_scatter:
            clear_output()

            x = selector_x.value
            y = selector_y.value
            color = selector_color.value

            if color == "Sin color":
                fig = px.scatter(
                    df,
                    x=x,
                    y=y,
                    title=f"Relación entre {x} y {y}",
                    opacity=0.7
                )
            else:
                fig = px.scatter(
                    df,
                    x=x,
                    y=y,
                    color=color,
                    title=f"Relación entre {x} y {y}, segmentado por {color}",
                    opacity=0.7
                )

            fig.show()

    selector_x.observe(dashboard_scatter, names="value")
    selector_y.observe(dashboard_scatter, names="value")
    selector_color.observe(dashboard_scatter, names="value")

    display(widgets.HBox([selector_x, selector_y, selector_color]), salida_scatter)
    dashboard_scatter()
else:
    print("Se necesitan al menos dos variables numéricas para crear un scatter plot.")


Output()

# 11. Mapa de correlación interactivo

La correlación ayuda a identificar relaciones lineales entre variables numéricas.

## Interpretación general

- Correlación cercana a `1`: relación positiva fuerte.
- Correlación cercana a `-1`: relación negativa fuerte.
- Correlación cercana a `0`: poca o ninguna relación lineal.

## Precaución

Correlación no significa causalidad.  
Que dos variables se muevan juntas no significa que una cause la otra.


In [18]:
# MAPA DE CORRELACIÓN

if len(variables_numericas) >= 2:
    corr = df[variables_numericas].corr()

    fig = px.imshow(
        corr,
        text_auto=True,
        title="Mapa de correlación entre variables numéricas",
        aspect="auto"
    )
    fig.show()
else:
    print("No hay suficientes variables numéricas para calcular correlaciones.")


# 12. Análisis interactivo por grupos

Esta sección permite agrupar una variable categórica y resumir una variable numérica.

## Controles

- Variable categórica para agrupar.
- Variable numérica para calcular.
- Métrica: promedio, mediana, suma, conteo, mínimo o máximo.
- Número de grupos a mostrar.

## Ejemplos de preguntas

- ¿Cuál es el promedio de ventas por región?
- ¿Cuál es el ingreso total por producto?
- ¿Qué categoría tiene más registros?
- ¿Cuál es la mediana del salario por cargo?


In [19]:
# ANÁLISIS INTERACTIVO POR GRUPOS

if len(variables_categoricas) > 0 and len(variables_numericas) > 0:
    selector_grupo = widgets.Dropdown(
        options=variables_categoricas,
        value=variables_categoricas[0],
        description="Grupo:"
    )

    selector_valor = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[0],
        description="Valor:"
    )

    selector_metrica = widgets.Dropdown(
        options=["mean", "median", "sum", "count", "min", "max"],
        value="mean",
        description="Métrica:"
    )

    slider_grupos = widgets.IntSlider(
        value=10,
        min=3,
        max=30,
        step=1,
        description="Top N:"
    )

    salida_grupos = widgets.Output()

    def dashboard_grupos(change=None):
        with salida_grupos:
            clear_output()

            grupo = selector_grupo.value
            valor = selector_valor.value
            metrica = selector_metrica.value
            top_n = slider_grupos.value

            tabla = (
                df.groupby(grupo)[valor]
                .agg(metrica)
                .sort_values(ascending=False)
                .head(top_n)
                .reset_index()
            )

            tabla.columns = [grupo, metrica]

            display(Markdown(f"## {metrica} de `{valor}` por `{grupo}`"))
            display(tabla)

            fig = px.bar(
                tabla,
                x=metrica,
                y=grupo,
                orientation="h",
                title=f"{metrica} de {valor} por {grupo}"
            )
            fig.update_layout(yaxis={"categoryorder": "total ascending"})
            fig.show()

    for widget in [selector_grupo, selector_valor, selector_metrica, slider_grupos]:
        widget.observe(dashboard_grupos, names="value")

    display(widgets.VBox([
        widgets.HBox([selector_grupo, selector_valor]),
        widgets.HBox([selector_metrica, slider_grupos])
    ]), salida_grupos)

    dashboard_grupos()
else:
    print("Se necesita al menos una variable categórica y una numérica para este análisis.")


Output()

# 13. Detección interactiva de outliers

Esta sección permite detectar valores atípicos usando el método IQR.

## ¿Qué es IQR?

IQR significa **Interquartile Range** o rango intercuartílico.

Se calcula como:

```text
IQR = Q3 - Q1
```

Donde:

- `Q1` es el percentil 25.
- `Q3` es el percentil 75.

Se consideran posibles outliers los valores que están por debajo o por encima de estos límites:

```text
Límite inferior = Q1 - 1.5 * IQR
Límite superior = Q3 + 1.5 * IQR
```


In [20]:
# DETECCIÓN INTERACTIVA DE OUTLIERS

def detectar_outliers_iqr(dataframe, columna):
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = dataframe[
        (dataframe[columna] < limite_inferior) |
        (dataframe[columna] > limite_superior)
    ]

    return outliers, limite_inferior, limite_superior

if len(variables_numericas) > 0:
    selector_outlier = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[0],
        description="Variable:"
    )

    salida_outliers = widgets.Output()

    def dashboard_outliers(change=None):
        with salida_outliers:
            clear_output()

            col = selector_outlier.value
            outliers, li, ls = detectar_outliers_iqr(df, col)

            display(Markdown(f"## Outliers para `{col}`"))
            print(f"Límite inferior: {li:.2f}")
            print(f"Límite superior: {ls:.2f}")
            print(f"Cantidad de outliers: {len(outliers)}")
            print(f"Porcentaje de outliers: {len(outliers) / len(df) * 100:.2f}%")

            fig = px.box(
                df,
                y=col,
                title=f"Boxplot con outliers para {col}"
            )
            fig.show()

            display(outliers.head(20))

    selector_outlier.observe(dashboard_outliers, names="value")

    display(selector_outlier, salida_outliers)
    dashboard_outliers()
else:
    print("No hay variables numéricas para detectar outliers.")


Dropdown(description='Variable:', options=('edad', 'tarifadiaria', 'distanciacasa', 'educacion', 'cantidadempl…

Output()

# 14. Análisis temporal interactivo

Esta sección se activa si el dataset tiene variables de fecha.

## ¿Para qué sirve?

Permite analizar cómo cambia una variable numérica a lo largo del tiempo.

Ejemplos:

- Ventas por mes.
- Ingresos por fecha.
- Cantidad de registros por día.
- Promedio de una variable por periodo.


In [21]:
# ANÁLISIS TEMPORAL INTERACTIVO

if len(variables_fecha) > 0 and len(variables_numericas) > 0:
    selector_fecha = widgets.Dropdown(
        options=variables_fecha,
        value=variables_fecha[0],
        description="Fecha:"
    )

    selector_valor_tiempo = widgets.Dropdown(
        options=variables_numericas,
        value=variables_numericas[0],
        description="Valor:"
    )

    selector_periodo = widgets.Dropdown(
        options={
            "Día": "D",
            "Mes": "M",
            "Año": "Y"
        },
        value="M",
        description="Periodo:"
    )

    selector_metrica_tiempo = widgets.Dropdown(
        options=["mean", "sum", "count", "median"],
        value="sum",
        description="Métrica:"
    )

    salida_tiempo = widgets.Output()

    def dashboard_tiempo(change=None):
        with salida_tiempo:
            clear_output()

            fecha = selector_fecha.value
            valor = selector_valor_tiempo.value
            periodo = selector_periodo.value
            metrica = selector_metrica_tiempo.value

            temp = df[[fecha, valor]].dropna().copy()
            temp["periodo"] = temp[fecha].dt.to_period(periodo).dt.to_timestamp()

            serie = (
                temp.groupby("periodo")[valor]
                .agg(metrica)
                .reset_index()
            )

            display(serie.head())

            fig = px.line(
                serie,
                x="periodo",
                y=valor,
                markers=True,
                title=f"{metrica} de {valor} por periodo"
            )
            fig.show()

    for widget in [selector_fecha, selector_valor_tiempo, selector_periodo, selector_metrica_tiempo]:
        widget.observe(dashboard_tiempo, names="value")

    display(widgets.VBox([
        widgets.HBox([selector_fecha, selector_valor_tiempo]),
        widgets.HBox([selector_periodo, selector_metrica_tiempo])
    ]), salida_tiempo)

    dashboard_tiempo()
else:
    print("No se detectaron variables de fecha y variables numéricas suficientes para el análisis temporal.")


No se detectaron variables de fecha y variables numéricas suficientes para el análisis temporal.


# 15. Panel para exponer la metodología CRISP-DM

Esta sección está diseñada para que los estudiantes escriban y presenten su metodología.

## ¿Cómo se usa?

Completa cada cuadro de texto con la explicación correspondiente.

Al final puedes usar esta información para una presentación, informe o exposición oral.


In [22]:
# PANEL INTERACTIVO DE METODOLOGÍA CRISP-DM
# IBM HR Analytics — Rotación de Personal


business = widgets.Textarea(
    value="""Problema: IBM enfrenta costos elevados asociados a la rotación voluntaria de personal.
Cada empleado que abandona la empresa genera costos de reemplazo estimados entre el 50% y el 200% del salario anual.

Objetivo: Identificar los factores más asociados a la rotación de empleados para diseñar
estrategias preventivas de retención de talento.

Preguntas clave:
- ¿Qué variables predicen mejor la rotación?
- ¿Qué departamentos tienen mayor tasa de rotación?
- ¿El ingreso mensual, las horas extra o la satisfacción laboral son determinantes?""",
    placeholder="Business Understanding",
    description="1. Negocio:",
    layout=widgets.Layout(width="100%", height="150px")
)

data_understanding = widgets.Textarea(
    value="""Dataset: IBM HR Analytics Employee Attrition & Performance
- Filas: 1,470 empleados
- Columnas: 36 variables
- Variable objetivo: Rotacion (Yes: 237 empleados = 16.1% / No: 1,233 = 83.9%)

Tipos de variables:
- Numéricas: 26 (edad, ingresos, satisfacción, años trabajados, etc.)
- Categóricas: 9 (departamento, género, estado civil, rol laboral, etc.)
- Sin variables de fecha en este dataset

Variables más relevantes identificadas:
IngresoMensual, NivelLaboral, HorasExtra, SatisfaccionLaboral,
EstadoCivil, ViajosNegocios, AniosEnEmpresa, Edad""",
    placeholder="Data Understanding",
    description="2. Datos:",
    layout=widgets.Layout(width="100%", height="150px")
)

data_preparation = widgets.Textarea(
    value="""Limpieza realizada:
Valores nulos: 0 (dataset completamente limpio)
Duplicados: 0 (sin registros repetidos)
Normalización de columnas: nombres en minúsculas con guión bajo
Variables eliminadas por ser constantes: HorasEstandar (80 para todos), MayorDe18 (Sí para todos), CantidadEmpleados (1 para todos)

Transformaciones:
- Columna 'Rotacion' usada como variable binaria (Yes/No)
- Variables de satisfacción tratadas como numéricas (escala 1-4)
- HorasExtra codificada como Yes/No""",
    placeholder="Data Preparation",
    description="3. Limpieza:",
    layout=widgets.Layout(width="100%", height="150px")
)

eda_findings = widgets.Textarea(
    value="""Principales hallazgos del análisis exploratorio:

1. TASA DE ROTACIÓN GENERAL: 16.1% (237 de 1,470 empleados)

2. FACTOR MÁS CRÍTICO — Horas Extra:
   - Empleados con horas extra: 30.5% de rotación
   - Empleados sin horas extra: 10.4% de rotación
   → 3x más riesgo con horas extra

3. INGRESO MENSUAL:
   - Empleados que rotaron: promedio $4,787/mes
   - Empleados que permanecieron: promedio $6,833/mes
   → Brecha de $2,046 mensual (43% más alto en quienes se quedan)

4. ROTACIÓN POR DEPARTAMENTO:
   - Ventas (Sales): 20.6% ← más alto
   - Recursos Humanos: 19.0%
   - Investigación & Desarrollo: 13.8%

5. ESTADO CIVIL:
   - Solteros: 25.5% de rotación
   - Casados: 12.5%
   - Divorciados: 10.1%

6. VIAJES DE NEGOCIOS:
   - Viajes frecuentes: 24.9% de rotación
   - Viajes esporádicos: 15.0%
   - Sin viajes: 8.0%

7. EDAD: Los empleados que rotan tienen en promedio 33.6 años vs 37.6 los que permanecen
   (empleados jóvenes son más propensos a rotar)

8. CORRELACIÓN INGRESO MENSUAL: Altamente correlacionado con NivelLaboral (0.95)
   y AniosTotalesTrabajo (0.77)""",
    placeholder="EDA Findings",
    description="4. EDA:",
    layout=widgets.Layout(width="100%", height="200px")
)

evaluation = widgets.Textarea(
    value="""El análisis responde satisfactoriamente las preguntas de negocio planteadas:

Se identificaron los principales factores de rotación
Las variables más discriminantes son: HorasExtra, IngresoMensual, NivelLaboral, EstadoCivil y ViajosNegocios
El dashboard permite explorar interactivamente cualquier combinación de variables

Limitaciones:
- El dataset no incluye variable de tiempo (fecha de contratación/salida), por lo que no se puede hacer análisis de supervivencia
- La variable objetivo está desbalanceada (16% Yes vs 84% No), lo que debe considerarse en modelos predictivos
- No se conoce el año de recolección de los datos

Próximo paso sugerido: construir un modelo de clasificación (Random Forest o XGBoost)
para predecir la probabilidad de rotación de cada empleado.""",
    placeholder="Evaluation",
    description="5. Evaluación:",
    layout=widgets.Layout(width="100%", height="150px")
)

deployment = widgets.Textarea(
    value="""Entregables de este proyecto:

1. Dashboard interactivo (este notebook) → compartir en Google Colab o GitHub
2. Resumen ejecutivo en Markdown → para documentación técnica
3. Presentación ejecutiva → para el equipo de People Analytics y directivos IBM

Recomendaciones estratégicas para IBM:
 1. Revisar política de horas extra — es el factor más fuerte de rotación
 2. Revisar estructura salarial en niveles 1-2 — empleados con menor ingreso rotan 3x más
 3. Programa de retención para empleados solteros y menores de 35 años
 4. Planes de carrera acelerados en el área de Ventas (mayor tasa de rotación: 20.6%)
 5. Revisar frecuencia de viajes de negocio para empleados con más de 2 años en la empresa""",
    placeholder="Deployment",
    description="6. Entrega:",
    layout=widgets.Layout(width="100%", height="150px")
)

display(Markdown("## 📋 Metodología CRISP-DM — IBM HR Analytics"))
display(Markdown("*Los campos ya contienen los hallazgos del análisis. Puedes editarlos antes de generar el resumen ejecutivo.*"))
display(business)
display(data_understanding)
display(data_preparation)
display(eda_findings)
display(evaluation)
display(deployment)


## 📋 Metodología CRISP-DM — IBM HR Analytics

*Los campos ya contienen los hallazgos del análisis. Puedes editarlos antes de generar el resumen ejecutivo.*

Textarea(value='Problema: IBM enfrenta costos elevados asociados a la rotación voluntaria de personal.\nCada e…

Textarea(value='Dataset: IBM HR Analytics Employee Attrition & Performance\n- Filas: 1,470 empleados\n- Column…

Textarea(value="Limpieza realizada:\nValores nulos: 0 (dataset completamente limpio)\nDuplicados: 0 (sin regis…

Textarea(value='Principales hallazgos del análisis exploratorio:\n\n1. TASA DE ROTACIÓN GENERAL: 16.1% (237 de…

Textarea(value='El análisis responde satisfactoriamente las preguntas de negocio planteadas:\n\nSe identificar…

Textarea(value='Entregables de este proyecto:\n\n1. Dashboard interactivo (este notebook) → compartir en Googl…

# 16. Generador de resumen ejecutivo

Esta sección toma los textos escritos en el panel anterior y genera un resumen estructurado.

## ¿Para qué sirve?

Sirve para que los estudiantes puedan copiar este resumen y usarlo en:

- Una exposición oral.
- Un informe escrito.
- Una presentación.
- Un README de GitHub.
- Un portafolio profesional.


In [23]:
# GENERADOR DE RESUMEN EJECUTIVO


boton_resumen = widgets.Button(
    description="Generar resumen ejecutivo",
    button_style="success"
)

salida_resumen = widgets.Output()

def generar_resumen(b):
    with salida_resumen:
        clear_output()

        resumen = f'''
# Resumen ejecutivo del proyecto

## 1. Business Understanding
{business.value}

## 2. Data Understanding
{data_understanding.value}

## 3. Data Preparation
{data_preparation.value}

## 4. EDA y hallazgos principales
{eda_findings.value}

## 5. Evaluation
{evaluation.value}

## 6. Deployment
{deployment.value}
'''
        display(Markdown(resumen))

boton_resumen.on_click(generar_resumen)

display(boton_resumen, salida_resumen)


Button(button_style='success', description='Generar resumen ejecutivo', style=ButtonStyle())

Output()

# 17. Exportar resumen metodológico a archivo Markdown

Esta sección permite guardar el resumen ejecutivo como archivo `.md`.

## Explicación

El archivo Markdown puede subirse a GitHub o convertirse en parte de la documentación del proyecto.


In [24]:
# EXPORTAR RESUMEN A MARKDOWN

boton_exportar = widgets.Button(
    description="Exportar resumen .md",
    button_style="info"
)

salida_exportar = widgets.Output()

def exportar_markdown(b):
    with salida_exportar:
        clear_output()

        contenido = f'''# Resumen ejecutivo del proyecto

## 1. Business Understanding
{business.value}

## 2. Data Understanding
{data_understanding.value}

## 3. Data Preparation
{data_preparation.value}

## 4. EDA y hallazgos principales
{eda_findings.value}

## 5. Evaluation
{evaluation.value}

## 6. Deployment
{deployment.value}
'''
        nombre_archivo = "resumen_ejecutivo_crispdm.md"

        with open(nombre_archivo, "w", encoding="utf-8") as f:
            f.write(contenido)

        print(f"Archivo creado: {nombre_archivo}")
        files.download(nombre_archivo)

boton_exportar.on_click(exportar_markdown)

display(boton_exportar, salida_exportar)


Button(button_style='info', description='Exportar resumen .md', style=ButtonStyle())

Output()

# 18. Guía para Exposición — IBM HR Analytics

Usa esta estructura para presentar el proyecto ante tus compañeros de IBM:

## 1. Introducción (1 min)
*"Este proyecto analiza el dataset de Recursos Humanos de IBM, compuesto por 1,470 empleados,
con el objetivo de identificar los factores que predicen la rotación voluntaria de personal."*

## 2. Problema de negocio (2 min)
*"La rotación de personal le cuesta a IBM entre el 50% y el 200% del salario anual por cada empleado que sale.
Con una tasa actual del 16.1% (237 empleados en este dataset), el impacto económico es significativo."*

## 3. Datos utilizados (1 min)
*"El dataset contiene 1,470 registros, 36 columnas y 0 valores nulos.
Las variables más importantes son: IngresoMensual, HorasExtra, NivelLaboral, SatisfaccionLaboral y EstadoCivil."*

## 4. Calidad de datos (1 min)
*"El dataset está completamente limpio: sin nulos, sin duplicados.
Se identificaron 3 variables constantes que fueron descartadas del análisis."*

## 5. Análisis exploratorio (5 min — mostrar el dashboard)
*"Durante el EDA exploramos distribuciones, correlaciones, outliers y relaciones entre variables."*
- Mostrar tarjetas de resumen (KPIs)
- Mostrar distribución de IngresoMensual por Rotación
- Mostrar gráfico de Rotación por HorasExtra
- Mostrar mapa de correlación

## 6. Hallazgos principales (3 min)

| # | Hallazgo | Impacto |
|---|----------|---------|
| 1 | Horas extra → 30.5% vs 10.4% rotación | 🔴 Alto |
| 2 | Brecha salarial: $4,787 vs $6,833 mensual | 🔴 Alto |
| 3 | Ventas: mayor tasa de rotación (20.6%) | 🟠 Medio |
| 4 | Solteros: 25.5% de rotación | 🟠 Medio |
| 5 | Viajes frecuentes: 24.9% de rotación | 🟠 Medio |

## 7. Recomendaciones (2 min)
1. 🎯 Revisar política de horas extra
2. 🎯 Ajustar estructura salarial en niveles 1-2
3. 🎯 Programa de retención para menores de 35 años
4. 🎯 Plan de carrera acelerado en Ventas
5. 🎯 Reducir frecuencia de viajes o compensación adicional

## 8. Próximos pasos
*"Como siguiente fase, se propone desarrollar un modelo predictivo de clasificación
(Random Forest o XGBoost) para identificar en tiempo real qué empleados tienen mayor
probabilidad de abandonar la empresa."*
